In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-mpnet-base-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


{'model_name': 'sentence-transformers/all-mpnet-base-v2', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 64, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


{'num_examples': 1500, 'columns': ['sentence1', 'sentence2', 'label']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   

                                  sentence2  label  
0      A man wearing a hard hat is dancing.   5.00  
1                A child is riding a horse.   4.75  
2  The man is feeding a mouse to the snake.   5.00  
3                  A man is playing guitar.   2.40  
4                 A man is playing a flute.   2.75  


In [3]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sentence-transformers/all-mpnet-base-v2


In [4]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)


In [5]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))


                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  cosine_similarity  \
0      A man wearing a hard hat is dancing.  5.000           0.996705   
1                A child is riding a horse.  4.750           0.950965   
2  The man is feeding a mouse to the snake.  5.000           0.854339   
3                  A man is playing guitar.  2.400           0.594890   
4                 A man is playing a flute.  2.750           0.735644   
5                  A man is cutting onions.  2.615           0.721671   
6       The man

In [6]:
top_k = 5

top_pred_pairs = results_df.nlargest(top_k, "predicted_score_0_5")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error"
]].reset_index(drop=True)

bottom_pred_pairs = results_df.nsmallest(top_k, "predicted_score_0_5")[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error"
]].reset_index(drop=True)

print("Top prediction pairs:")
print(top_pred_pairs.to_string(index=False))
print()
print("Bottom prediction pairs:")
print(bottom_pred_pairs.to_string(index=False))


Top prediction pairs:
                                      sentence1                                       sentence2  label  predicted_score_0_5  cosine_similarity  absolute_error
Colorado Governor Visits School Shooting Victim Colorado governor visits school shooting victim    5.0             5.000000           1.000000        0.000000
              A man with a hard hat is dancing.            A man wearing a hard hat is dancing.    5.0             4.991763           0.996705        0.008237
            Matt Smith quits BBC‚Äôs Doctor Who               Matt Smith quits BBC's Doctor Who    5.0             4.987588           0.995035        0.012412
                          People are near water                          People are near water.    5.0             4.986464           0.994586        0.013536
            There are people out on the street.                   People are out on the street.    5.0             4.975223           0.990089        0.024777

Bottom prediction pairs

In [7]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")


device_used: mps
model_name: sentence-transformers/all-mpnet-base-v2
dataset_split: glue/stsb/validation
num_examples: 1500
spearman_correlation: 0.881091
pearson_correlation: 0.880625
predicted_score_mean: 3.916513
predicted_score_std: 0.689590
label_mean: 2.363908
label_std: 1.499985
runtime_seconds: 23.30
